In [3]:
import json
import os
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
REPO_ROOT        = Path(__file__).parents[2] if "__file__" in dir() else Path("../../")
MOVIES_FILE      = REPO_ROOT / "saved_imdb_movies.json"
EVALUATIONS_DIR  = Path("../../evaluations")

# The 7 non-anchor metadata attributes (one per vector space)
METADATA_ATTRS = [
    "plot_events_metadata",
    "plot_analysis_metadata",
    "viewer_experience_metadata",
    "watch_context_metadata",
    "narrative_techniques_metadata",
    "reception_metadata",
    "production_metadata",
]

VARIANT_KEY = "original_full_tokens"

# ── Load movies ───────────────────────────────────────────────────────────────
with open(MOVIES_FILE) as f:
    movies: list[dict] = json.load(f)

print(f"Loaded {len(movies)} movies")

# ── Build per-attribute evaluation dicts ─────────────────────────────────────
# Structure: { attr_name: { imdb_id: { variant: metadata_json } } }
evaluation_data: dict[str, dict] = {attr: {} for attr in METADATA_ATTRS}

for movie in movies:
    imdb_id = movie["id"]
    for attr in METADATA_ATTRS:
        metadata = movie.get(attr)
        if metadata is None:
            # Skip movies that don't have this attribute populated yet
            continue
        # Initialise the movie entry if this is the first variant we're writing
        if imdb_id not in evaluation_data[attr]:
            evaluation_data[attr][imdb_id] = {}
        evaluation_data[attr][imdb_id][VARIANT_KEY] = metadata

# ── Persist to ./evaluations/<attr>.json ─────────────────────────────────────
EVALUATIONS_DIR.mkdir(parents=True, exist_ok=True)

for attr, data in evaluation_data.items():
    out_path = EVALUATIONS_DIR / f"{attr}.json"
    with open(out_path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"Wrote {len(data):>3} movies → {out_path}")


Loaded 50 movies
Wrote  50 movies → ../../evaluations/plot_events_metadata.json
Wrote  50 movies → ../../evaluations/plot_analysis_metadata.json
Wrote  50 movies → ../../evaluations/viewer_experience_metadata.json
Wrote  50 movies → ../../evaluations/watch_context_metadata.json
Wrote  50 movies → ../../evaluations/narrative_techniques_metadata.json
Wrote  50 movies → ../../evaluations/reception_metadata.json
Wrote  50 movies → ../../evaluations/production_metadata.json


In [13]:
import sqlite3

# Path to the tracker DB (relative to this notebook at implementation/notebooks/)
TRACKER_DB_PATH = REPO_ROOT / "ingestion_data" / "tracker.db"

with sqlite3.connect(TRACKER_DB_PATH) as db:
    # Count affected rows before deletion for confirmation
    count = db.execute("""
        SELECT COUNT(*)
        FROM tmdb_data
        WHERE tmdb_id IN (
            SELECT tmdb_id FROM movie_progress WHERE status = 'filtered_out'
        )
    """).fetchone()[0]

    db.execute("""
        DELETE FROM tmdb_data
        WHERE tmdb_id IN (
            SELECT tmdb_id FROM movie_progress WHERE status = 'filtered_out'
        )
    """)
    db.commit()

print(f"Deleted {count:,} rows from tmdb_data where movie_progress.status = 'filtered_out'")


Deleted 22,825 rows from tmdb_data where movie_progress.status = 'filtered_out'
